# Step 1 — Audit the raw dataset

#### Imports

In [1]:
import pandas as pd
import numpy as np

#### Load

In [2]:
path = '../data/raw/scrapped_data.csv'
real_estate = pd.read_csv(path)
real_estate.head()

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Dimensions,Under_Construction
0,3 BHK Flat for Sale in Saarrthi Shimmer n Shi...,1.75 Cr,"11,305 per sqft",1165 sqft,Ready to Move,6 out of 12,Resale,Furnished,East,Main Road,Saarrthi Shimmer n Shine,Co-operative Society,2 Covered,3,2.0,NaN,NaN,NaN
1,"3 BHK Flat for Sale in Acolade, Kharadi, Pune",1.52 Cr,"8,444 per sqft",1400 sqft,Ready to Move,7 out of 12,Resale,Semi-Furnished,East,Garden/Park,Acolade,Freehold,2 Covered,3,2.0,NaN,NaN,NaN
2,"4 BHK Flat for Sale in Windsor Avenue, Wanowr...",2.60 Cr,"11,064 per sqft",NaN,Ready to Move,4 out of 9,Resale,Semi-Furnished,East,Main Road,Windsor Avenue,Co-operative Society,2 Covered,5,3.0,2350 sqft,NaN,NaN
3,"1 BHK Flat for Sale in Raheja Sterling, NIBM ...",61.6 Lac,"8,107 per sqft",560 sqft,Ready to Move,1 out of 18,Resale,Unfurnished,East,"Garden/Park, Main Road",Raheja Sterling,Co-operative Society,1 Covered,1,NaN,NaN,NaN,NaN
4,"3 BHK House for Sale in Kalpana Empress, Wano...",2.75 Cr,"13,750 per sqft",NaN,Ready to Move,NaN,Resale,Furnished,East,Main Road,Kalpana Empress,Co-operative Society,3 Covered,4,2.0,2000 sqft,NaN,NaN


#### Get the overall structure

In [3]:
real_estate.info()

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               930 non-null    str    
 1   Price               930 non-null    str    
 2   Rate_per_SqFt       820 non-null    str    
 3   Carpet_Area         846 non-null    str    
 4   Status              735 non-null    str    
 5   Floor               770 non-null    str    
 6   Transaction         930 non-null    str    
 7   Furnishing          929 non-null    str    
 8   facing              776 non-null    str    
 9   overlooking         756 non-null    str    
 10  Society             811 non-null    str    
 11  Ownership           776 non-null    str    
 12  Car_Parking         554 non-null    str    
 13  Bathroom            930 non-null    int64  
 14  Balcony             867 non-null    float64
 15  Super_Area          84 non-null     str    
 16  Dimensions         

In [4]:
real_estate.isna().sum()

Title                   0
Price                   0
Rate_per_SqFt         110
Carpet_Area            84
Status                195
Floor                 160
Transaction             0
Furnishing              1
facing                154
overlooking           174
Society               119
Ownership             154
Car_Parking           376
Bathroom                0
Balcony                63
Super_Area            846
Dimensions            906
Under_Construction    749
dtype: int64

In [5]:
real_estate.duplicated().sum()

np.int64(3)

Duplicate records

In [6]:
real_estate[real_estate.duplicated(keep=False)]

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Dimensions,Under_Construction
144,"4 BHK Apartment for Sale in Godrej Ivara, Khar...",3.09 Cr,NaN,1550 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Godrej Ivara,NaN,NaN,4,4.0,NaN,NaN,Poss. by Dec '31
162,"4 BHK Apartment for Sale in Godrej Ivara, Khar...",3.09 Cr,NaN,1550 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Godrej Ivara,NaN,NaN,4,4.0,NaN,NaN,Poss. by Dec '31
184,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29
185,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29
186,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29


These are ducplicate rows (duplicate listings)
so retaining just one copy is enough. 

In [7]:
real_estate.duplicated().sum(), real_estate.shape

(np.int64(3), (930, 18))

In [8]:
real_estate = real_estate.drop_duplicates(keep='first')

In [9]:
real_estate.duplicated().sum(), real_estate.shape

(np.int64(0), (927, 18))

Missing value percentage per column

In [10]:
pd.concat([real_estate.isna().sum().rename('NaN_cnt'), round(real_estate.isna().mean() * 100, 2).rename('PerMissing_pct')], axis = 1).sort_values(by = 'PerMissing_pct', ascending= False)

,NaN_cnt,PerMissing_pct
Dimensions,903,97.41
Super_Area,843,90.94
Under_Construction,749,80.80
Car_Parking,373,40.24
Status,192,20.71
overlooking,171,18.45
Floor,157,16.94
facing,151,16.29
Ownership,151,16.29
Society,119,12.84


### hidden missing values

In [11]:
missing_tokens = ["", "nan", "none", "n/a", "null", "-"]

text_df = real_estate.select_dtypes(include="str")
sus_counts = text_df.apply(
    lambda col: col.str.strip().str.lower().isin(missing_tokens).sum()
)
sus_counts


Title                 0
Price                 0
Rate_per_SqFt         0
Carpet_Area           0
Status                0
Floor                 0
Transaction           0
Furnishing            0
facing                0
overlooking           0
Society               0
Ownership             0
Car_Parking           0
Super_Area            0
Dimensions            0
Under_Construction    0
dtype: int64

#### Selecting Columns to Drop.

1. Dimensions
97.42% missing

In [12]:
real_estate.Dimensions.value_counts(dropna= False).head()

Dimensions
NaN                  903
40 X 50 ft Sqft        2
45 X 28 ft Sqft        1
80 X 50 ft Sqft        1
100 X 115 ft Sqft      1
Name: count, dtype: int64

Insights
- seems mostly consistent in format (length * width)
- not in directly usable numeric format
- after removing duplicates NaN value count is 903

Do the values provide information that isn't already captured by Carpet_Area or Super_Area?

In [13]:
real_estate.loc[
    real_estate.Dimensions.notna(),
    ['Dimensions', 'Carpet_Area', 'Super_Area']
].head(10)

,Dimensions,Carpet_Area,Super_Area
6,45 X 28 ft Sqft,2400 sqft,NaN
120,80 X 50 ft Sqft,3000 sqft,NaN
128,100 X 115 ft Sqft,2000 sqft,NaN
241,40 X 50 ft Sqft,2825 sqft,NaN
307,78 X 22.44 ft Sqft,5440 sqft,NaN
335,1000 X 1.5 ft Sqft,1600 sqft,NaN
340,45 X 55 ft Sqft,3000 sqft,NaN
341,50 X 42.98 ft Sqft,2965 sqft,NaN
370,50 X 60 ft Sqft,5000 sqft,NaN
378,60 X 58.33 ft Sqft,1800 sqft,NaN


Insight
- Super_Area is missing for 22 of them
- Dimensions don't appear consistent with other Area values. For example, 45 × 28 = 1,260 square feet, while the corresponding Carpet_Area is 2,400 sqft


##### Decision
Drop Dimensions from this cleaning project<br>
Justification :
- It contains approx 97% missing values
- Remaining values contains inconsistent text instead of numeric measurement
- Interpretation relative to other area columns is uncertain

In [14]:
real_estate = real_estate.drop(columns= 'Dimensions')
real_estate.head()

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Under_Construction
0,3 BHK Flat for Sale in Saarrthi Shimmer n Shi...,1.75 Cr,"11,305 per sqft",1165 sqft,Ready to Move,6 out of 12,Resale,Furnished,East,Main Road,Saarrthi Shimmer n Shine,Co-operative Society,2 Covered,3,2.0,NaN,NaN
1,"3 BHK Flat for Sale in Acolade, Kharadi, Pune",1.52 Cr,"8,444 per sqft",1400 sqft,Ready to Move,7 out of 12,Resale,Semi-Furnished,East,Garden/Park,Acolade,Freehold,2 Covered,3,2.0,NaN,NaN
2,"4 BHK Flat for Sale in Windsor Avenue, Wanowr...",2.60 Cr,"11,064 per sqft",NaN,Ready to Move,4 out of 9,Resale,Semi-Furnished,East,Main Road,Windsor Avenue,Co-operative Society,2 Covered,5,3.0,2350 sqft,NaN
3,"1 BHK Flat for Sale in Raheja Sterling, NIBM ...",61.6 Lac,"8,107 per sqft",560 sqft,Ready to Move,1 out of 18,Resale,Unfurnished,East,"Garden/Park, Main Road",Raheja Sterling,Co-operative Society,1 Covered,1,NaN,NaN,NaN
4,"3 BHK House for Sale in Kalpana Empress, Wano...",2.75 Cr,"13,750 per sqft",NaN,Ready to Move,NaN,Resale,Furnished,East,Main Road,Kalpana Empress,Co-operative Society,3 Covered,4,2.0,2000 sqft,NaN


In [15]:
real_estate.shape

(927, 17)

### Investigate Super_Area

In [16]:
real_estate.Super_Area.value_counts(dropna=False).head()

Super_Area
NaN          843
1500 sqft      4
1600 sqft      2
1775 sqft      2
1650 sqft      2
Name: count, dtype: int64

- Has 843 missing values after removing duplicates

In [17]:
real_estate.loc[
    real_estate.Super_Area.notna(),
    ['Super_Area', 'Carpet_Area']
# ]
].head(10)

,Super_Area,Carpet_Area
2,2350 sqft,NaN
4,2000 sqft,NaN
9,2845 sqft,NaN
11,3200 sqft,NaN
18,1500 sqft,NaN
33,780 sqft,NaN
38,1045 sqft,NaN
89,1092 sqft,NaN
99,3000 sqft,NaN
105,1600 sqft,NaN


Observation / Pattern
- Every row containing `Super_Area` has a missing `Carpet_Area`.
- Two columns may be alternative ways of reporting a property's area in this dataset.

#### Make the decision

In [18]:
real_estate.loc[
    real_estate.Super_Area.notna() & real_estate.Carpet_Area.notna(),
    ['Super_Area', 'Carpet_Area']
]

,Super_Area,Carpet_Area


In [19]:
real_estate.loc[
    real_estate.Super_Area.isna() & real_estate.Carpet_Area.isna(),
    ['Super_Area', 'Carpet_Area']
]

,Super_Area,Carpet_Area


That gives us a clean partition:
- Every row has Super_Area populated, or
- Every row has Carpet_Area populated,
- But never both and never neither.

Decision: <br>
For now lets keep it as it is .
<br>
Later, after typecasting to numeric values, we can consider creating a unified analytical column with a clearly documented fallback rule.

### Investigate `Under_Construction` column

In [20]:
real_estate.Under_Construction.value_counts(dropna= False).head()

Under_Construction
NaN                 749
Poss. by Dec '27     25
Poss. by Dec '26     22
Poss. by Dec '28     22
Poss. by Dec '29     13
Name: count, dtype: int64

Observation
- 749 missing values.
- The remaining records contain possession estimates such as Poss. by Dec '27, Poss. by Dec '26, and Poss. by Dec '30.
- These values appear to represent expected possession dates, not simply a Yes/No construction status. <br> <br>


This column may tell us when a property is expected to be ready for possession, which could be useful when analyzing prices and availability

#### Decision
Keep it for now<br>

Reason:
- Contains expected possession dates
- Could help distinguish listings with future possession dates
- Losing potentially useful timing information

### final audit before Step 2

In [21]:
missing_tokens = ["", "nan", "none", "n/a", "null", "-"]

real_estate.select_dtypes(include="str").apply(
    lambda col: col.str.strip().str.lower().isin(missing_tokens).sum()
).sum()

np.int64(0)

---

# Step 2 — String cleaning and type-casting

### Price

In [22]:
real_estate.Price.value_counts().head()

Price
1.70 Cr    20
1.15 Cr    19
1.25 Cr    17
1.10 Cr    16
85 Lac     15
Name: count, dtype: int64

In [23]:
real_estate.Price.value_counts().tail()

Price
67.4 Lac    1
2.95 Cr     1
74 Lac      1
53.2 Lac    1
2.82 Cr     1
Name: count, dtype: int64

It contains values in two format Cr and Lac

1 Cr = ₹10,000,000

1 Lac = ₹100,000

##### Function to convert these Price values in numeric format

In [24]:
def price_cleaner(price: str) -> float:
    n, unit = price.strip().split()

    if unit == "Cr":
        multiplier = 10_000_000
    elif unit == "Lac":
        multiplier = 100_000
    else:
        raise ValueError(f"Unknown price unit: {unit}")
    
    return float(n) * multiplier

In [25]:
print(price_cleaner("1.70 Cr"))   # 17000000.0
print(price_cleaner("85 Lac"))    # 8500000.0
print(price_cleaner("61.6 Lac"))  # 6160000.0
price_cleaner("85  Lac")   # 8500000.0

17000000.0
8500000.0
6160000.0


8500000.0

Testing function

In [26]:
temp_clead_prices = pd.DataFrame({
    "Original"  : real_estate.Price,
    "Cleaned" : real_estate.Price.map(price_cleaner)
})
temp_clead_prices

,Original,Cleaned
0,1.75 Cr,17500000.0
1,1.52 Cr,15200000.0
2,2.60 Cr,26000000.0
3,61.6 Lac,6160000.0
4,2.75 Cr,27500000.0
...,...,...
925,49.5 Lac,4950000.0
926,1.80 Cr,18000000.0
927,1.20 Cr,12000000.0
928,1.80 Cr,18000000.0


In [27]:
temp_clead_prices.Cleaned.isna().sum()

np.int64(0)

In [28]:
temp_clead_prices.loc[[temp_clead_prices.Cleaned.idxmin(), temp_clead_prices.Cleaned.idxmax()], : ]

,Original,Cleaned
741,14 Lac,1400000.0
89,16.38 Cr,163800000.0


In [29]:
real_estate["Price"] = temp_clead_prices["Cleaned"]
real_estate.Price.dtype, real_estate.Price.isna().sum()

(dtype('float64'), np.int64(0))

### Rate_per_SqFt

In [30]:
real_estate.Rate_per_SqFt.value_counts().head()

Rate_per_SqFt
10,000 per sqft    17
12,000 per sqft     8
7,500 per sqft      8
8,333 per sqft      6
14,000 per sqft     4
Name: count, dtype: int64

Data is in `10,000 per sqft` format.
We need to convet it into numeric rs ie float values

In [31]:
real_estate["Rate_per_SqFt"].isna().sum()

np.int64(107)

Function to clean `Rate_per_SqFt`

In [32]:
def rate_cleaner(old_rate):
    if pd.isna(old_rate):
        return old_rate
    return float(old_rate.removesuffix(' per sqft').replace(',', ''))
rate_cleaner(np.nan), rate_cleaner('10,000 per sqft')

(nan, 10000.0)

Testing rate_cleaner

In [33]:
temp_rate = pd.DataFrame({
    'old_rate' : real_estate.Rate_per_SqFt,
    'new_rate' : real_estate['Rate_per_SqFt'].map(rate_cleaner)
})
temp_rate.new_rate.isna().sum(), temp_rate.new_rate.dtype

(np.int64(107), dtype('float64'))

In [34]:
temp_rate.loc[
    [temp_rate.new_rate.idxmin(), temp_rate.new_rate.idxmax()], 
    :
]

,old_rate,new_rate
562,"2,333 per sqft",2333.0
89,"1,50,000 per sqft",150000.0


In [35]:
temp_rate.new_rate.describe()

count       820.000000
mean      10746.668293
std        6225.845656
min        2333.000000
25%        7990.750000
50%       10000.000000
75%       12860.500000
max      150000.000000
Name: new_rate, dtype: float64

The median is lower than the mean, suggesting the distribution may be right-skewed. The maximum of ₹1,50,000 is particularly far above the 75th percentile.

In [36]:
150000.0/12860.5

11.663621165584543

##### inspect the highest rates in Rate_per_SqFt

In [37]:
temp_rate.nlargest(5, 'new_rate')

,old_rate,new_rate
89,"1,50,000 per sqft",150000.0
457,"26,727 per sqft",26727.0
99,"26,633 per sqft",26633.0
141,"26,338 per sqft",26338.0
112,"25,000 per sqft",25000.0


In [38]:
real_estate.loc[
    temp_rate.nlargest(5, 'new_rate').index,
    :
]

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Under_Construction
89,3 BHK Apartment for Sale in Majestique Evolvus...,163800000.0,"1,50,000 per sqft",NaN,NaN,2 out of 26,New Property,Semi-Furnished,NaN,NaN,Majestique Evolvus,NaN,NaN,3,1.0,1092 sqft,Poss. by Dec '28
457,3 BHK Apartment for Sale in Bhoslenagar Ashok ...,73500000.0,"26,727 per sqft",2050 sqft,Ready to Move,10 out of 27,Resale,Furnished,West,Garden/Park,NaN,Freehold,2 Covered,4,2.0,NaN,NaN
99,"3 BHK Apartment for Sale in Vascon Windermere,...",79900000.0,"26,633 per sqft",NaN,Ready to Move,17 out of 22,Resale,Semi-Furnished,East,NaN,Vascon Windermere,Co-operative Society,NaN,3,2.0,3000 sqft,NaN
141,4 BHK Apartment for Sale in Venkatesh Erandwan...,43900000.0,"26,338 per sqft",1670 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Venkatesh Erandwane Central,NaN,NaN,4,3.0,NaN,Poss. by Sep '26
112,"4 BHK Apartment for Sale in Marvel Ribera, Bun...",92500000.0,"25,000 per sqft",2600 sqft,Ready to Move,7 out of 14,Resale,Semi-Furnished,East,NaN,Marvel Ribera,Freehold,3 Covered,5,1.0,NaN,NaN


#### Investigating an Extreme Price-per-SqFt Outlier

During outlier analysis, row **89** was identified as an extreme value in the `Rate_per_SqFt` column.

- **Listed price:** ₹16.38 Cr
- **Rate per sq. ft.:** ₹1,50,000
- **Property:** 3 BHK apartment in Majestique Evolvus, Pune
- **Reported super area:** 1,092 sq. ft.

The rate is significantly higher than the next-highest observation in the dataset, making this record worth investigating before deciding how to handle it.

##### External Reference

Project details were compared with the [Majestique Evolvus project listing on SquareYards](https://www.squareyards.com/pune-residential-property/majestique-evolvus/240133/project).

The page reports a project asking rate of approximately ₹15,647 per sq. ft. and a project price range of ₹1.15 Cr–₹2.69 Cr. These figures suggest that the dataset's ₹1,50,000 per sq. ft. value may be anomalous.

##### Possible Explanation

One hypothesis is that the listed price contains an extra zero. If the price were ₹1.638 Cr instead of ₹16.38 Cr, dividing it by the reported super area of 1,092 sq. ft. would give approximately ₹15,000 per sq. ft., which is much closer to the project's reported rate.

However, the external project page does **not** verify the exact price of this specific listing. Therefore, this correction remains a hypothesis rather than a confirmed fact.

##### Cleaning Decision

Do not silently modify the price or rate based only on this comparison. Preserve the original evidence and flag the record for further verification. If the exact listing cannot be verified, retain the original values or exclude the record from a particular analysis only under a clearly documented rule.

**Conclusion:** Row 89 is a potential data-quality issue that requires investigation, not an automatically confirmed error.

In [39]:
real_estate['Rate_per_SqFt'] = temp_rate['new_rate']
del temp_rate

In [40]:
real_estate.Rate_per_SqFt.info()

<class 'pandas.Series'>
Index: 927 entries, 0 to 929
Series name: Rate_per_SqFt
Non-Null Count  Dtype  
--------------  -----  
820 non-null    float64
dtypes: float64(1)
memory usage: 46.8 KB


In [41]:
real_estate["Rate_per_SqFt"].describe()

count       820.000000
mean      10746.668293
std        6225.845656
min        2333.000000
25%        7990.750000
50%       10000.000000
75%       12860.500000
max      150000.000000
Name: Rate_per_SqFt, dtype: float64

### Carpet_Area Cleaning

In [42]:
real_estate.Carpet_Area.head(10)

0    1165 sqft
1    1400 sqft
2          NaN
3     560 sqft
4          NaN
5    1006 sqft
6    2400 sqft
7    1000 sqft
8     730 sqft
9          NaN
Name: Carpet_Area, dtype: str

In [43]:
real_estate.Carpet_Area.isna().sum(), real_estate.Carpet_Area.dtype

(np.int64(84), <StringDtype(storage='python', na_value=nan)>)

In [44]:
temp_carpet_area = pd.DataFrame({
    'Old' : real_estate.Carpet_Area,
    'Cleaned' : real_estate['Carpet_Area'].map(lambda x : x if pd.isna(x) else int(x.removesuffix(' sqft')))
})
temp_carpet_area.head()

,Old,Cleaned
0,1165 sqft,1165.0
1,1400 sqft,1400.0
2,NaN,NaN
3,560 sqft,560.0
4,NaN,NaN


In [45]:
temp_carpet_area["Cleaned"].dtype, temp_carpet_area["Cleaned"].isna().sum(), temp_carpet_area["Cleaned"].describe()

(dtype('float64'),
 np.int64(84),
 count      843.000000
 mean      1216.943060
 std        953.538616
 min        100.000000
 25%        806.000000
 50%       1055.000000
 75%       1325.000000
 max      21000.000000
 Name: Cleaned, dtype: float64)

In [46]:
temp_carpet_area.nlargest(10, "Cleaned")

,Old,Cleaned
873,21000 sqft,21000.0
386,6980 sqft,6980.0
307,5440 sqft,5440.0
876,5250 sqft,5250.0
370,5000 sqft,5000.0
469,4800 sqft,4800.0
804,4282 sqft,4282.0
130,4095 sqft,4095.0
901,3950 sqft,3950.0
179,3895 sqft,3895.0


In [47]:
temp_carpet_area.nsmallest(10, "Cleaned")

,Old,Cleaned
741,100 sqft,100.0
725,255 sqft,255.0
138,258 sqft,258.0
293,258 sqft,258.0
84,320 sqft,320.0
586,326 sqft,326.0
608,336 sqft,336.0
733,400 sqft,400.0
577,412 sqft,412.0
746,416 sqft,416.0


- Row 873 — 21,000 sqft: an extreme value compared with the median of 1,055 sqft. It could be a data-entry error or a genuinely large property.
- Row 741 — 100 sqft: unusually small, but not enough evidence to call it incorrect.

In [48]:
real_estate.loc[
    [873, 741],
    ["Title", "Price", "Rate_per_SqFt", "Carpet_Area",
     "Status", "Floor", "Society", "Super_Area"]
]

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Society,Super_Area
873,"3 BHK Villa for Sale in Bhugaon, Bavdhan Pune",75000000.0,3571.0,21000 sqft,Ready to Move,NaN,NaN,NaN
741,1 BHK Apartment for Sale in Bhawani Peth Pune,1400000.0,8750.0,100 sqft,Ready to Move,3 out of 4,NaN,NaN


#### Flagging Suspicious Carpet Area Values

During the outlier investigation, two records were identified as potentially problematic based on their carpet area and property details.

| Row Index | Property | Carpet Area | Price | Rate per SqFt |
|---:|---|---:|---:|---:|
| 873 | 3 BHK Villa, Bhugaon, Bavdhan | 21,000 sqft | ₹7.5 Cr | ₹3,571 |
| 741 | 1 BHK Apartment, Bhawani Peth | 100 sqft | ₹14 Lakh | ₹8,750 |

**Row 873 — Unusually Large Area**

The reported carpet area of 21,000 sqft is unusually large for a 3 BHK villa. However, the listed price divided by the carpet area is approximately ₹3,571 per sqft, consistent with the recorded rate. The value may be genuine or may reflect a data-entry issue.

**Row 741 — Unusually Small Area**

The reported carpet area of 100 sqft is unusually small for a 1 BHK apartment. Additionally, the listed price of ₹14 lakh divided by the carpet area gives ₹14,000 per sqft, which does not match the recorded rate of ₹8,750 per sqft.

#### Decision

Both records are **flagged for further investigation**, not automatically classified as errors.

- Preserve the original values until reliable evidence is available.
- Do not replace suspicious values with estimates without verification.
- Consider excluding these records from specific analyses only if a documented, justified rule requires it.

**Conclusion:** These observations highlight potential data-quality issues. Further verification is needed before correcting or removing either record.

In [49]:
real_estate["Carpet_Area"] = temp_carpet_area["Cleaned"]

In [50]:
real_estate["Carpet_Area"].dtype, real_estate["Carpet_Area"].isna().sum(), real_estate["Carpet_Area"].describe()

(dtype('float64'),
 np.int64(84),
 count      843.000000
 mean      1216.943060
 std        953.538616
 min        100.000000
 25%        806.000000
 50%       1055.000000
 75%       1325.000000
 max      21000.000000
 Name: Carpet_Area, dtype: float64)

### real_estate.

In [51]:
real_estate.Floor.value_counts(dropna= False).head(20)

Floor
NaN             157
7 out of 12      17
5 out of 12      12
5 out of 11      12
8 out of 11      11
4 out of 11      10
10 out of 12     10
5 out of 7       10
3 out of 11      10
3 out of 12       9
1 out of 12       9
6 out of 12       8
2 out of 7        8
4 out of 12       8
9 out of 11       7
9 out of 12       7
12 out of 12      7
7 out of 14       7
4 out of 7        7
8 out of 12       7
Name: count, dtype: int64

In [52]:
real_estate.loc[
    real_estate.Floor.notna(), 
    'Floor'
].sample(
    10, random_state= 42
)

821     5 out of 12
912     1 out of 12
885     2 out of 12
728    13 out of 31
895    10 out of 16
323      4 out of 4
133     7 out of 11
750      3 out of 6
715    10 out of 12
264      2 out of 3
Name: Floor, dtype: str

Seems consistent format

In [53]:
temp_floor = pd.DataFrame({
    'Old_Floor' : real_estate.Floor
})
temp_floor[['Floor_Number', 'Total_Floors']] =  real_estate.Floor.str.split(" out of ", expand= True)
temp_floor.head()

,Old_Floor,Floor_Number,Total_Floors
0,6 out of 12,6,12
1,7 out of 12,7,12
2,4 out of 9,4,9
3,1 out of 18,1,18
4,NaN,NaN,NaN


In [54]:
temp_floor[['Floor_Number', 'Total_Floors']] = temp_floor[['Floor_Number', 'Total_Floors']].apply(
    pd.to_numeric, errors = 'coerce' 
)

In [55]:
temp_floor.head()

,Old_Floor,Floor_Number,Total_Floors
0,6 out of 12,6.0,12.0
1,7 out of 12,7.0,12.0
2,4 out of 9,4.0,9.0
3,1 out of 18,1.0,18.0
4,NaN,NaN,NaN


In [56]:
temp_floor.dtypes, temp_floor.isna().sum()

(Old_Floor           str
 Floor_Number    float64
 Total_Floors    float64
 dtype: object,
 Old_Floor       157
 Floor_Number    170
 Total_Floors    157
 dtype: int64)

Floor_Number has 13 more missing values than Total_Floors

In [57]:
temp_floor.loc[
    (temp_floor.Floor_Number.isna()) & (temp_floor.Total_Floors.notna())
]

,Old_Floor,Floor_Number,Total_Floors
129,Upper Basement out of 27,NaN,27.0
132,Ground out of 5,NaN,5.0
139,Ground out of 6,NaN,6.0
226,Ground out of 19,NaN,19.0
338,Lower Basement out of 3,NaN,3.0
455,Upper Basement out of 19,NaN,19.0
480,Upper Basement out of 20,NaN,20.0
527,Upper Basement out of 10,NaN,10.0
534,Upper Basement out of 11,NaN,11.0
670,Ground out of 3,NaN,3.0


##### Special cases
- Ground out of 5 — ground floor, 5 total floors.
- Upper Basement out of 27 — upper basement, 27 total floors.
- Lower Basement out of 3 — lower basement, 3 total floors.

#### Decision
Create a new column named `Floor_Label` to preserve the original floor description while keeping the Floor_Number numeric for analysis.

In [58]:
temp_floor['Floor_Label'] = temp_floor.Old_Floor.str.extract(r'^(Ground|Upper Basement|Lower Basement)')

In [59]:
temp_floor.loc[
    temp_floor.Floor_Label == 'Ground',
    'Floor_Number'
] = 0

In [60]:
temp_floor["Floor_Label"].value_counts(dropna=False).head()

Floor_Label
NaN               914
Ground              7
Upper Basement      5
Lower Basement      1
Name: count, dtype: int64

In [61]:
temp_floor.isna().sum()

Old_Floor       157
Floor_Number    163
Total_Floors    157
Floor_Label     914
dtype: int64

Insight
- Floor_Number: 163 missing = 157 originally missing + 6 basement floors.
- Total_Floors: 157 missing, unchanged from the original.
- Floor_Label: 914 missing because only 13 records have special text labels.

Valadite Floor values

In [62]:
temp_floor[["Floor_Number", "Total_Floors"]].describe()

,Floor_Number,Total_Floors
count,764.000000,770.000000
mean,7.685864,15.079221
std,5.851886,8.142119
min,0.000000,1.000000
25%,3.000000,10.000000
50%,7.000000,12.000000
75%,10.000000,21.000000
max,44.000000,45.000000


In [63]:
temp_floor.loc[
    temp_floor["Floor_Number"].notna()
    & (temp_floor["Floor_Number"] > temp_floor["Total_Floors"]),
    ["Old_Floor", "Floor_Number", "Total_Floors"]
]

,Old_Floor,Floor_Number,Total_Floors


In [64]:
real_estate[["Floor_Number", "Total_Floors", "Floor_Label"]] = (
    temp_floor[["Floor_Number", "Total_Floors", "Floor_Label"]]
)
del temp_floor

In [65]:
real_estate[
    ["Floor", "Floor_Number", "Total_Floors", "Floor_Label"]
].head(10)

,Floor,Floor_Number,Total_Floors,Floor_Label
0,6 out of 12,6.0,12.0,NaN
1,7 out of 12,7.0,12.0,NaN
2,4 out of 9,4.0,9.0,NaN
3,1 out of 18,1.0,18.0,NaN
4,NaN,NaN,NaN,NaN
5,6 out of 22,6.0,22.0,NaN
6,NaN,NaN,NaN,NaN
7,4 out of 11,4.0,11.0,NaN
8,7 out of 21,7.0,21.0,NaN
9,13 out of 32,13.0,32.0,NaN


### Cleaning Status

In [66]:
real_estate.Status.value_counts(dropna= False).head()

Status
Ready to Move    735
NaN              192
Name: count, dtype: int64

contains only one non-missing value.

#### Decision: 
No cleaning needed. Keep the column unchanged.

### Furnishing

In [67]:
real_estate["Furnishing"].value_counts(dropna=False)

Furnishing
Unfurnished       498
Semi-Furnished    297
Furnished         131
NaN                 1
Name: count, dtype: int64

Seems clean.

contains only three Catagories and just one missing value

#### Decision: 
Keep Furnishing unchanged. The labels are already consistent, and there's no reason to modify the single missing value.

### facing

In [68]:
real_estate["facing"].value_counts(dropna=False)

facing
East            592
NaN             151
West             62
North            56
North - East     39
South            10
South - East      7
North - West      5
South -West       5
Name: count, dtype: int64

Values seems clean

just the spacing of direction label is inconsistent.

Fix

In [69]:
# real_estate.facing.str.replace(' ', '').value_counts()

In [70]:
real_estate["facing"] = real_estate["facing"].str.replace(
    r"\s*-\s*", "-", regex=True
)

In [71]:
real_estate["facing"].value_counts(dropna=False)

facing
East          592
NaN           151
West           62
North          56
North-East     39
South          10
South-East      7
North-West      5
South-West      5
Name: count, dtype: int64

#### Decision:
facing is cleaned.

### real_state.`overlooking`

In [72]:
real_estate.overlooking.value_counts(dropna= False)

overlooking
Garden/Park, Pool, Main Road    178
NaN                             171
Main Road                       149
Garden/Park, Main Road          148
Garden/Park                     146
Garden/Park, Pool                61
Pool, Garden/Park, Main Road     42
Pool                              9
Pool, Garden/Park                 8
Main Road, Garden/Park, Pool      7
Main Road, Garden/Park            6
Pool, Main Road                   2
Name: count, dtype: int64

#### Split into separate indicator columns — 
- create Overlooks_Garden_Park, Overlooks_Pool, and Overlooks_Main_Road, each containing True or False
- with missing data handled separately.

##### We'll create three Boolean columns:
- Overlooks_Garden_Park
- Overlooks_Pool
- Overlooks_Main_Road

In [73]:
real_estate['overlooking'] = real_estate['overlooking'].astype('string')

we convert it to 'string' as 

before it contained np.nan instead pd.NA

In [74]:
# Your exact code
real_estate["Overlooks_Garden_Park"] = (
    real_estate["overlooking"]
    .str.contains("Garden/Park", regex=False, na=pd.NA)
)

# If you check the value counts of your NEW column:
print(real_estate["Overlooks_Garden_Park"].value_counts(dropna=False))

Overlooks_Garden_Park
True     596
<NA>     171
False    160
Name: count, dtype: Int64


In [75]:
real_estate.Overlooks_Garden_Park.dtype

BooleanDtype

In [76]:
real_estate["Overlooks_Pool"] = (
    real_estate["overlooking"]
    .str.contains("Pool", regex=False, na=pd.NA)
)

real_estate["Overlooks_Main_Road"] = (
    real_estate["overlooking"]
    .str.contains("Main Road", regex=False, na=pd.NA)
)

In [77]:
real_estate[
    [
        "overlooking",
        "Overlooks_Garden_Park",
        "Overlooks_Pool",
        "Overlooks_Main_Road",
    ]
].head(10)

,overlooking,Overlooks_Garden_Park,Overlooks_Pool,Overlooks_Main_Road
0,Main Road,False,False,True
1,Garden/Park,True,False,False
2,Main Road,False,False,True
3,"Garden/Park, Main Road",True,False,True
4,Main Road,False,False,True
5,Main Road,False,False,True
6,"Pool, Garden/Park, Main Road",True,True,True
7,Garden/Park,True,False,False
8,"Garden/Park, Main Road",True,False,True
9,"Garden/Park, Pool",True,True,False


In [78]:
real_estate[
    [
        'Overlooks_Garden_Park',
        'Overlooks_Pool',
        'Overlooks_Main_Road'
    ]
].apply(lambda col : col.value_counts(dropna = False))

,Overlooks_Garden_Park,Overlooks_Pool,Overlooks_Main_Road
False,160,449,224
True,596,307,532
<NA>,171,171,171


#### Final decision for overlooking
Keep the original column for now. The three new indicators are more useful for analysis, but retaining the source column preserves traceability. We can decide whether to drop it during the final review.

### Society

In [79]:
real_estate["Society"].value_counts(dropna=False).head(20)

Society
NaN                          119
Panchshil Towers               8
Kolte Patil Life Republic      7
Godrej Elaris                  6
Sukhwani Skylines              6
Panchshil Eon Waterfront       5
Paranjape Blue Ridge           5
Amanora Elevate Towers         5
Goel Ganga Serio               5
Bramha the Collection          5
Goel Ganga Platino             5
Lodha Bella Vita               4
Pristine The Lords             4
Unique K Shire                 4
Godrej Evergreen Square        4
Skyi Songbirds                 4
Godrej Green Cove              4
Marvel Cerise                  4
Shubh Nirvana                  3
Raheja Reserve                 3
Name: count, dtype: int64

In [80]:
# check whitespace and capitalization
real_estate.loc[
    real_estate["Society"].notna()
    & (real_estate["Society"] != real_estate["Society"].str.strip()),
    "Society"
]

Series([], Name: Society, dtype: str)

In [81]:
real_estate["Society"].dropna().groupby(
    real_estate["Society"].dropna().str.casefold()
).nunique().loc[lambda x: x > 1]

Series([], Name: Society, dtype: int64)

- No leading or trailing whitespace was found in Society.
- No society names differing only by capitalization were detected.
- The 119 missing values should remain missing.

#### Decision: Society needs no cleaning. Keep it unchanged.


### real_state.`Ownership`

In [82]:
real_estate["Ownership"].value_counts(dropna=False)

Ownership
Freehold                523
Co-operative Society    230
NaN                     151
Leasehold                22
Power Of Attorney         1
Name: count, dtype: int64

Column is already clean.

#### Decision: Keep the values unchanged and preserve the 151 missing values.

### real_estate.`Car_Parking`

In [83]:
real_estate.Car_Parking.value_counts(dropna= False)

Car_Parking
NaN            373
1 Covered      297
2 Covered      118
1 Covered,      48
1 Open          29
3 Covered       22
2 Covered,      14
4 Covered        8
2 Open           3
10 Covered       2
3 Covered,       2
101 Covered      1
4 Covered,       1
6 Covered        1
24 Covered       1
35 Covered       1
303 Covered      1
604 Covered      1
703 Open         1
5 Open           1
603 Open         1
205 Open         1
Name: count, dtype: int64

1. Inconsistent formatting: values such as 1 Covered and 1 Covered, should be standardized.
2. Suspicious counts: values such as 101 Covered, 303 Covered, and 703 Open may represent malformed data rather than actual parking counts.

In [84]:
sorted(real_estate["Car_Parking"].dropna().unique())

['1 Covered',
 '1 Covered,',
 '1 Open',
 '10 Covered',
 '101 Covered',
 '2 Covered',
 '2 Covered,',
 '2 Open',
 '205 Open',
 '24 Covered',
 '3 Covered',
 '3 Covered,',
 '303 Covered',
 '35 Covered',
 '4 Covered',
 '4 Covered,',
 '5 Open',
 '6 Covered',
 '603 Open',
 '604 Covered',
 '703 Open']

In [85]:
real_estate.loc[
    real_estate["Car_Parking"].str.contains(
        r"^(10|24|35|101|205|303|603|604|703)",
        regex=True,
        na=False
    ),
    ["Title", "Car_Parking", "Price", "Society"]
]

/tmp/ipykernel_13856/1381641393.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  real_estate["Car_Parking"].str.contains(


,Title,Car_Parking,Price,Society
251,"2 BHK Apartment for Sale in Bramha Avenue, Kon...",10 Covered,8800000.0,Bramha Avenue
296,"3 BHK Apartment for Sale in Tennessee, Talegao...",101 Covered,10000000.0,Tennessee
631,4 BHK Apartment for Sale in Sukhsagar Nagar Ka...,24 Covered,14000000.0,NaN
719,"2 BHK Apartment for Sale in Kumar Paradise, Ha...",35 Covered,12000000.0,Kumar Paradise
738,2 BHK Apartment for Sale in Balewadi Pune,303 Covered,10800000.0,NaN
761,2 BHK Apartment for Sale in Wellwisher Kiarah ...,10 Covered,8000000.0,Wellwisher Kiarah Terrazo Phase II
765,"2 BHK Apartment for Sale in Kasturi Eon Homes,...",604 Covered,11300000.0,Kasturi Eon Homes
798,"2 BHK Apartment for Sale in Royal Imperio, Pim...",703 Open,13000000.0,Royal Imperio
917,2 BHK Apartment for Sale in Aishwaryam Venture...,603 Open,7400000.0,Aishwaryam Ventures
929,"1 BHK Apartment for Sale in White Lily, Vishnu...",205 Open,6700000.0,White Lily


In [86]:
real_estate['Car_Parking'] = real_estate.Car_Parking.str.strip().str.rstrip(',').str.strip()
real_estate["Car_Parking"].value_counts(dropna=False)

Car_Parking
NaN            373
1 Covered      345
2 Covered      132
1 Open          29
3 Covered       24
4 Covered        9
2 Open           3
10 Covered       2
101 Covered      1
6 Covered        1
24 Covered       1
35 Covered       1
303 Covered      1
604 Covered      1
703 Open         1
5 Open           1
603 Open         1
205 Open         1
Name: count, dtype: int64

### Bathroom

In [87]:
real_estate["Bathroom"].value_counts(dropna=False).sort_index()

Bathroom
1     55
2    371
3    360
4    104
5     33
6      2
7      1
8      1
Name: count, dtype: int64

#### Decision: 
Keep Bathroom unchanged as an integer column.

### Balcony

In [88]:
real_estate["Balcony"].value_counts(dropna=False).sort_index()

Balcony
1.0    240
2.0    413
3.0    185
4.0     20
5.0      3
6.0      3
NaN     63
Name: count, dtype: int64

clean

### Super_Area

In [89]:
real_estate["Super_Area"].head(10)

0          NaN
1          NaN
2    2350 sqft
3          NaN
4    2000 sqft
5          NaN
6          NaN
7          NaN
8          NaN
9    2845 sqft
Name: Super_Area, dtype: str

In [90]:
real_estate["Super_Area"].isna().sum()

np.int64(843)

In [91]:
temp_super_area = pd.DataFrame({
    'Old' : real_estate.Super_Area,
    'Cleaned' : real_estate.Super_Area.map(
        lambda x : x if pd.isna(x) 
        else int(x.removesuffix(' sqft'))
    )
})
temp_super_area.head()

,Old,Cleaned
0,NaN,NaN
1,NaN,NaN
2,2350 sqft,2350.0
3,NaN,NaN
4,2000 sqft,2000.0


In [92]:
temp_super_area["Cleaned"].dtype, temp_super_area["Cleaned"].isna().sum()

(dtype('float64'), np.int64(843))

In [93]:
temp_super_area["Cleaned"].describe()

count      84.000000
mean     1661.285714
std      1129.855557
min       400.000000
25%       916.250000
50%      1342.500000
75%      1887.500000
max      5300.000000
Name: Cleaned, dtype: float64

In [94]:
real_estate["Super_Area"] = temp_super_area["Cleaned"]
del temp_super_area

### Under_Construction

In [95]:
real_estate["Under_Construction"].value_counts(dropna=False).head(20)

Under_Construction
NaN                 749
Poss. by Dec '27     25
Poss. by Dec '26     22
Poss. by Dec '28     22
Poss. by Dec '29     13
Poss. by Dec '30      9
Poss. by Jun '27      7
Poss. by Mar '29      7
Poss. by Oct '30      5
Poss. by May '30      5
Poss. by Dec '31      4
Poss. by Mar '28      4
Poss. by Jun '29      4
Poss. by Jun '26      4
Poss. by Sep '26      4
Poss. by Sep '28      4
Poss. by Apr '30      3
Poss. by Apr '27      3
Poss. by Jun '30      3
Poss. by Aug '30      3
Name: count, dtype: int64

We won't assume a missing value means the property is ready to move.

Keep dates as text for now.

In [96]:
real_estate.info()

<class 'pandas.DataFrame'>
Index: 927 entries, 0 to 929
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Title                  927 non-null    str    
 1   Price                  927 non-null    float64
 2   Rate_per_SqFt          820 non-null    float64
 3   Carpet_Area            843 non-null    float64
 4   Status                 735 non-null    str    
 5   Floor                  770 non-null    str    
 6   Transaction            927 non-null    str    
 7   Furnishing             926 non-null    str    
 8   facing                 776 non-null    str    
 9   overlooking            756 non-null    string 
 10  Society                808 non-null    str    
 11  Ownership              776 non-null    str    
 12  Car_Parking            554 non-null    str    
 13  Bathroom               927 non-null    int64  
 14  Balcony                864 non-null    float64
 15  Super_Area            

In [97]:
real_estate.dtypes

Title                        str
Price                    float64
Rate_per_SqFt            float64
Carpet_Area              float64
Status                       str
Floor                        str
Transaction                  str
Furnishing                   str
facing                       str
overlooking               string
Society                      str
Ownership                    str
Car_Parking                  str
Bathroom                   int64
Balcony                  float64
Super_Area               float64
Under_Construction           str
Floor_Number             float64
Total_Floors             float64
Floor_Label                  str
Overlooks_Garden_Park    boolean
Overlooks_Pool           boolean
Overlooks_Main_Road      boolean
dtype: object

In [98]:
assert real_estate.shape == (927, 23)
assert real_estate["Price"].notna().all()
assert real_estate["Rate_per_SqFt"].isna().sum() == 107
assert real_estate["Carpet_Area"].isna().sum() == 84
assert real_estate["Super_Area"].isna().sum() == 843
assert real_estate["Overlooks_Garden_Park"].isna().sum() == 171

---

## Step 3 — Missing-Value Handling and Column Selection.

### Audit missing values

In [99]:
missing_summary = pd.DataFrame({
    'Missing_Count' : real_estate.isna().sum(),
    'Missing_Percentage' : real_estate.isna().mean() * 100
})
missing_summary = missing_summary.sort_values('Missing_Count', ascending = False)
missing_summary

,Missing_Count,Missing_Percentage
Floor_Label,914,98.597627
Super_Area,843,90.938511
Under_Construction,749,80.798274
Car_Parking,373,40.237325
Status,192,20.711974
overlooking,171,18.446602
Overlooks_Garden_Park,171,18.446602
Overlooks_Main_Road,171,18.446602
Overlooks_Pool,171,18.446602
Floor_Number,163,17.583603


In [100]:
real_estate.columns

Index(['Title', 'Price', 'Rate_per_SqFt', 'Carpet_Area', 'Status', 'Floor',
       'Transaction', 'Furnishing', 'facing', 'overlooking', 'Society',
       'Ownership', 'Car_Parking', 'Bathroom', 'Balcony', 'Super_Area',
       'Under_Construction', 'Floor_Number', 'Total_Floors', 'Floor_Label',
       'Overlooks_Garden_Park', 'Overlooks_Pool', 'Overlooks_Main_Road'],
      dtype='str')

### Floor_Label

In [101]:
real_estate.loc[
    real_estate.Floor_Label.notna(),
    ['Floor', 'Super_Area', 'Total_Floors', 'Floor_Label', 'Floor_Number']
]

,Floor,Super_Area,Total_Floors,Floor_Label,Floor_Number
129,Upper Basement out of 27,NaN,27.0,Upper Basement,NaN
132,Ground out of 5,NaN,5.0,Ground,0.0
139,Ground out of 6,NaN,6.0,Ground,0.0
226,Ground out of 19,NaN,19.0,Ground,0.0
338,Lower Basement out of 3,NaN,3.0,Lower Basement,NaN
455,Upper Basement out of 19,NaN,19.0,Upper Basement,NaN
480,Upper Basement out of 20,NaN,20.0,Upper Basement,NaN
527,Upper Basement out of 10,1250.0,10.0,Upper Basement,NaN
534,Upper Basement out of 11,NaN,11.0,Upper Basement,NaN
670,Ground out of 3,NaN,3.0,Ground,0.0


We can consider dropping the original Floor column later, once we're satisfied these features preserve the information we need.

### Super_Area

In [102]:
# Super_Area present   False  True 
# Carpet_Area present              
# False                    0    843
# True                    84      0


pd.crosstab(
    real_estate.Carpet_Area.isna(),
    real_estate.Super_Area.isna(),
    rownames=["Carpet_Area present"],
    colnames=["Super_Area present"]
)

Super_Area present,False,True
Carpet_Area present,,
False,0,843
True,84,0


Even though Super_Area is 90.9% missing, it provides information for 843 listings that have no Carpet_Area value. Dropping it would leave those listings without either area measurement.

We will combine these later

### Under_Construction

In [103]:
pd.crosstab(
    real_estate["Status"],
    real_estate["Under_Construction"].isna(),
    margins=True
)

Under_Construction,True,All
Status,,
Ready to Move,735,735
All,735,735


Insight: 
- All 735 records with a non-missing Status value (Ready to Move) have missing values in Under_Construction.

In [104]:
pd.crosstab(
    real_estate.Status.fillna('Missing Status'),
    real_estate.Under_Construction.notna(),
    margins = True
)

Under_Construction,False,True,All
Status,,,
Missing Status,14,178,192
Ready to Move,735,0,735
All,749,178,927


Insight :

- All 735 properties marked Ready to Move have missing Under_Construction values. This is consistent with the idea that possession-date information is generally unnecessary for ready-to-move properties.
- 178 properties have possession-date information, and all 178 have missing Status. These dates may help describe when under-construction properties are expected to be available.
- 14 properties have both fields missing. We don't have enough information to infer their status.

They appear to complement each other:
- Status identifies ready-to-move properties.
- Under_Construction provides possession timing for 178 properties.

#### Final decision for `Floor`

In [105]:
real_estate.columns

Index(['Title', 'Price', 'Rate_per_SqFt', 'Carpet_Area', 'Status', 'Floor',
       'Transaction', 'Furnishing', 'facing', 'overlooking', 'Society',
       'Ownership', 'Car_Parking', 'Bathroom', 'Balcony', 'Super_Area',
       'Under_Construction', 'Floor_Number', 'Total_Floors', 'Floor_Label',
       'Overlooks_Garden_Park', 'Overlooks_Pool', 'Overlooks_Main_Road'],
      dtype='str')

In [106]:
real_estate.loc[
     : ,
    ['Floor', 'Floor_Label', 'Total_Floors', 'Floor_Number']
].sample(10, random_state=42)

,Floor,Floor_Label,Total_Floors,Floor_Number
166,NaN,NaN,NaN,NaN
433,9 out of 11,NaN,11.0,9.0
30,4 out of 11,NaN,11.0,4.0
67,25 out of 27,NaN,27.0,25.0
752,20 out of 30,NaN,30.0,20.0
886,4 out of 9,NaN,9.0,4.0
365,6 out of 7,NaN,7.0,6.0
107,5 out of 28,NaN,28.0,5.0
317,5 out of 11,NaN,11.0,5.0
295,4 out of 12,NaN,12.0,4.0


In [107]:
real_estate = real_estate.drop(columns= 'Floor')

### `overlooking`

In [108]:
real_estate[
    ["overlooking", "Overlooks_Garden_Park",
     "Overlooks_Pool", "Overlooks_Main_Road"]
].head(10)

,overlooking,Overlooks_Garden_Park,Overlooks_Pool,Overlooks_Main_Road
0,Main Road,False,False,True
1,Garden/Park,True,False,False
2,Main Road,False,False,True
3,"Garden/Park, Main Road",True,False,True
4,Main Road,False,False,True
5,Main Road,False,False,True
6,"Pool, Garden/Park, Main Road",True,True,True
7,Garden/Park,True,False,False
8,"Garden/Park, Main Road",True,False,True
9,"Garden/Park, Pool",True,True,False


In [109]:
real_estate = real_estate.drop(columns='overlooking')

### `Floor_Label`

It has 13 non-missing values, but those values are meaningful and would be lost if we dropped the column.

In [110]:
real_estate.shape, real_estate.columns.tolist()

((927, 21),
 ['Title',
  'Price',
  'Rate_per_SqFt',
  'Carpet_Area',
  'Status',
  'Transaction',
  'Furnishing',
  'facing',
  'Society',
  'Ownership',
  'Car_Parking',
  'Bathroom',
  'Balcony',
  'Super_Area',
  'Under_Construction',
  'Floor_Number',
  'Total_Floors',
  'Floor_Label',
  'Overlooks_Garden_Park',
  'Overlooks_Pool',
  'Overlooks_Main_Road'])

### Task 3 — Choose how to handle missing values

In [111]:
real_estate[
    [
        "Rate_per_SqFt",
        "Carpet_Area",
        "Super_Area",
        "Floor_Number",
        "Total_Floors",
        "Balcony",
    ]
].describe()

,Rate_per_SqFt,Carpet_Area,Super_Area,Floor_Number,Total_Floors,Balcony
count,820.000000,843.000000,84.000000,764.000000,770.000000,864.000000
mean,10746.668293,1216.943060,1661.285714,7.685864,15.079221,2.006944
std,6225.845656,953.538616,1129.855557,5.851886,8.142119,0.819772
min,2333.000000,100.000000,400.000000,0.000000,1.000000,1.000000
25%,7990.750000,806.000000,916.250000,3.000000,10.000000,1.000000
50%,10000.000000,1055.000000,1342.500000,7.000000,12.000000,2.000000
75%,12860.500000,1325.000000,1887.500000,10.000000,21.000000,2.000000
max,150000.000000,21000.000000,5300.000000,44.000000,45.000000,6.000000


| Column          | Missing | Recommendation             | Reason                                                                                                    |
| --------------- | ------- | -------------------------- | --------------------------------------------------------------------------------------------------------- |
| `Rate_per_SqFt` | 107     | Preserve for now           | Rates may be derivable from price and area, but area definitions and outlier values need care.            |
| `Carpet_Area`   | 84      | Preserve for now           | Missing values are complemented by `Super_Area` in this dataset, but the measurements are not equivalent. |
| `Super_Area`    | 843     | Preserve for now           | It provides area information for 843 listings that lack carpet area.                                      |
| `Floor_Number`  | 163     | Preserve                   | Missing values may include basements and listings with no floor information.                              |
| `Total_Floors`  | 157     | Preserve                   | Missing values cannot safely be inferred from other rows.                                                 |
| `Balcony`       | 63      | Consider median imputation | It is a numeric count, but first consider whether missing means unknown rather than a typical count.      |

### Balcony

In [112]:
real_estate.Transaction.value_counts(dropna= False)

Transaction
Resale          668
New Property    259
Name: count, dtype: int64

In [113]:
real_estate.groupby(by = 'Transaction', dropna = False)['Balcony'].agg(['count', 'mean', 'median'])

,count,mean,median
Transaction,,,
New Property,229,2.187773,2.0
Resale,635,1.941732,2.0


Close mean and median

this doesn't tell us what the missing values actually mean

Decision : eave the 63 missing Balcony values as NaN

In [114]:
real_estate.nunique(dropna=True).sort_values()

Status                     1
Transaction                2
Overlooks_Garden_Park      2
Overlooks_Pool             2
Overlooks_Main_Road        2
Floor_Label                3
Furnishing                 3
Ownership                  4
Balcony                    6
Bathroom                   8
facing                     8
Car_Parking               17
Floor_Number              32
Under_Construction        37
Total_Floors              43
Super_Area                74
Price                    323
Carpet_Area              450
Society                  619
Rate_per_SqFt            687
Title                    846
dtype: int64

| Column(s)            | Decision            | Reason                                      |
| -------------------- | ------------------- | ------------------------------------------- |
| `Balcony`            | Keep missing values | The true counts are unknown.                |
| `Rate_per_SqFt`      | Keep missing values | Avoid unreliable derived values.            |
| `Carpet_Area`        | Keep missing values | Don't invent area measurements.             |
| `Super_Area`         | Keep missing values | Useful for listings without carpet area.    |
| `Floor_Number`       | Keep missing values | Some floors are unknown or basement labels. |
| `Total_Floors`       | Keep missing values | Total floors cannot be safely inferred.     |
| `Status`             | Keep                | Preserves explicitly recorded status.       |
| `Under_Construction` | Keep                | Possession information is useful.           |
| `Floor_Label`        | Keep                | Preserves ground and basement distinctions. |

In [115]:
real_estate.isna().sum().sort_values(ascending= False)

Floor_Label              914
Super_Area               843
Under_Construction       749
Car_Parking              373
Status                   192
Overlooks_Garden_Park    171
Overlooks_Pool           171
Overlooks_Main_Road      171
Floor_Number             163
Total_Floors             157
facing                   151
Ownership                151
Society                  119
Rate_per_SqFt            107
Carpet_Area               84
Balcony                   63
Furnishing                 1
Price                      0
Title                      0
Bathroom                   0
Transaction                0
dtype: int64

| Column(s)            | Decision            | Reason                                      |
| -------------------- | ------------------- | ------------------------------------------- |
| `Balcony`            | Keep missing values | The true counts are unknown.                |
| `Rate_per_SqFt`      | Keep missing values | Avoid unreliable derived values.            |
| `Carpet_Area`        | Keep missing values | Don't invent area measurements.             |
| `Super_Area`         | Keep missing values | Useful for listings without carpet area.    |
| `Floor_Number`       | Keep missing values | Some floors are unknown or basement labels. |
| `Total_Floors`       | Keep missing values | Total floors cannot be safely inferred.     |
| `Status`             | Keep                | Preserves explicitly recorded status.       |
| `Under_Construction` | Keep                | Possession information is useful.           |
| `Floor_Label`        | Keep                | Preserves ground and basement distinctions. |

In [116]:
print("Shape:", real_estate.shape)

print("\nDuplicate rows:", real_estate.duplicated().sum())

print("\nData types:")
print(real_estate.dtypes)

print("\nMissing values:")
print(real_estate.isna().sum().sort_values(ascending=False))

Shape: (927, 21)

Duplicate rows: 0

Data types:
Title                        str
Price                    float64
Rate_per_SqFt            float64
Carpet_Area              float64
Status                       str
Transaction                  str
Furnishing                   str
facing                       str
Society                      str
Ownership                    str
Car_Parking                  str
Bathroom                   int64
Balcony                  float64
Super_Area               float64
Under_Construction           str
Floor_Number             float64
Total_Floors             float64
Floor_Label                  str
Overlooks_Garden_Park    boolean
Overlooks_Pool           boolean
Overlooks_Main_Road      boolean
dtype: object

Missing values:
Floor_Label              914
Super_Area               843
Under_Construction       749
Car_Parking              373
Status                   192
Overlooks_Garden_Park    171
Overlooks_Pool           171
Overlooks_Main_Road     

---

## Step 4. Feature Engineering

In [117]:
real_estate['Title'].sample(25, random_state= 42)

166    3 BHK Apartment for Sale in VTP MONARQUE, Hinj...
433    3 BHK Apartment for Sale in ARV Regalia, NIBM ...
30     2 BHK Apartment for Sale in Jasminum, Magarpat...
67     3 BHK Apartment for Sale in GK Sindhu Skies, P...
752    4 BHK Apartment for Sale in Livience Aleenta, ...
886    2 BHK Apartment for Sale in Laxmi Nagar Balewa...
365    3 BHK Penthouse for Sale in Lunkad Orchids, Cl...
107    4 BHK Apartment for Sale in ANP Autograph, Pun...
317    3 BHK Apartment for Sale in Rachana Bella Casa...
295    2 BHK Apartment for Sale in Eiffel City Phase ...
63     2 BHK Apartment for Sale in Sukhwani Skylines,...
602        2 BHK Apartment for Sale in Olark, Moshi Pune
855               4 BHK Villa for Sale in Erandwane Pune
721    1 BHK Apartment for Sale in Vedant Kingston Se...
539    2 BHK Apartment for Sale in Jasminum, Jasminiu...
238    3 BHK Apartment for Sale in Montvert Avion, Pa...
697    2 BHK Apartment for Sale in Punawale, Pimpri C...
869    2 BHK Apartment for Sale

### BHK

In [118]:
real_estate['Title'].str.extract(r'(\d+)\s*BHK').isna().sum()

0    0
dtype: int64

In [119]:
real_estate['BHK'] = (
    real_estate.Title
    .str.extract(r'(\d+)\s*BHK')[0]
    .astype('Int16')
)

In [120]:
real_estate["BHK"].value_counts(dropna=False).sort_index()

BHK
1     60
2    328
3    399
4    140
Name: count, dtype: Int64

### Property_Type

In [121]:
real_estate['Property_Type'] = real_estate.Title.str.extract(r'BHK\s+(.+?)\s+for Sale')

In [122]:
real_estate[["Title", "Property_Type"]].sample(15, random_state=42)

,Title,Property_Type
166,"3 BHK Apartment for Sale in VTP MONARQUE, Hinj...",Apartment
433,"3 BHK Apartment for Sale in ARV Regalia, NIBM ...",Apartment
30,"2 BHK Apartment for Sale in Jasminum, Magarpat...",Apartment
67,"3 BHK Apartment for Sale in GK Sindhu Skies, P...",Apartment
752,"4 BHK Apartment for Sale in Livience Aleenta, ...",Apartment
886,2 BHK Apartment for Sale in Laxmi Nagar Balewa...,Apartment
365,"3 BHK Penthouse for Sale in Lunkad Orchids, Cl...",Penthouse
107,"4 BHK Apartment for Sale in ANP Autograph, Pun...",Apartment
317,3 BHK Apartment for Sale in Rachana Bella Casa...,Apartment
295,2 BHK Apartment for Sale in Eiffel City Phase ...,Apartment


In [123]:
real_estate.Property_Type.value_counts(dropna= False)

Property_Type
Apartment        834
Villa             35
Flat              28
House             15
Penthouse         11
Builder Floor      4
Name: count, dtype: int64

In [124]:
real_estate.Title.sample(25)

393    3 BHK Apartment for Sale in Pride World City W...
921    2 BHK Apartment for Sale in Shagun Perfect 10 ...
472                3 BHK Apartment for Sale in Camp Pune
624    1 BHK Apartment for Sale in Verdura, Dudulgaon...
628    2 BHK Apartment for Sale in TCG The Cliff Gard...
824    2 BHK Apartment for Sale in Rachana Bella Casa...
442    3 BHK Apartment for Sale in Parth Eurus, Pasha...
423    3 BHK Apartment for Sale in Nanded City, Sinhg...
322    3 BHK Apartment for Sale in Ganga Liviano, Kha...
61     4 BHK Apartment for Sale in Bramhacorp Sun Val...
131    4 BHK Apartment for Sale in Unique K Shire, Pu...
286    4 BHK Apartment for Sale in Godrej Ivara, Khar...
177    3 BHK Apartment for Sale in Nivasa Enchante, L...
238    3 BHK Apartment for Sale in Montvert Avion, Pa...
100    2 BHK Apartment for Sale in Duville Riverdale ...
134    2 BHK Apartment for Sale in Rohan Harita, Tath...
513    2 BHK Apartment for Sale in Jhamtani Ace Atmos...
132    3 BHK Apartment for Sale

### Locality

| Title                                             | Locality                                        |
| ------------------------------------------------- | ----------------------------------------------- |
| 3 BHK Apartment for Sale in VTP MONARQUE, Hinj... | Hinjewadi (if confirmed by the full title)      |
| 2 BHK Apartment for Sale in Olark, Moshi Pune     | Moshi                                           |
| 4 BHK Villa for Sale in Erandwane Pune            | Erandwane                                       |
| 2 BHK Builder Floor for Sale in Narhe Road Pune   | Narhe Road                                      |
| 3 BHK Apartment for Sale in ARV Regalia, NIBM ... | NIBM area (exact locality needs the full title) |

In [125]:
real_estate.Title.loc[real_estate['Title'].str.split(',').str[-2].isna()]

101       2 BHK Apartment for Sale in Galande Nagar Pune
102            2 BHK Apartment for Sale in Kohinoor Pune
128             3 BHK Villa for Sale in Gold Valley Pune
151               2 BHK Apartment for Sale in Undri Pune
227    2 BHK Apartment for Sale in Phase 2 Hinjewadi ...
                             ...                        
860           3 BHK Apartment for Sale in Erandwane Pune
862    3 BHK Apartment for Sale in Mayur Colony Road ...
878       3 BHK Apartment for Sale in Kaspate Wasti Pune
883                 3 BHK Villa for Sale in Wagholi Pune
886    2 BHK Apartment for Sale in Laxmi Nagar Balewa...
Name: Title, Length: 79, dtype: str

In [126]:
real_estate["Title"].str.contains(",", regex=False).value_counts()

Title
True     848
False     79
Name: count, dtype: int64

In [127]:
with pd.option_context("display.max_colwidth", None):
    display(
        real_estate.loc[
            ~real_estate["Title"].str.contains(",", regex=False),
            ["Title"]
        ].head(20)
    )

,Title
101,2 BHK Apartment for Sale in Galande Nagar Pune
102,2 BHK Apartment for Sale in Kohinoor Pune
128,3 BHK Villa for Sale in Gold Valley Pune
151,2 BHK Apartment for Sale in Undri Pune
227,2 BHK Apartment for Sale in Phase 2 Hinjewadi Rajiv Gandhi Infotech Park Pune
239,3 BHK Apartment for Sale in Sector Number 6 Moshi Pune
247,2 BHK Apartment for Sale in Mohan Nagar Co operative Society Pune
260,1 BHK Apartment for Sale in Fursungi Bhekrai Road Pune
267,2 BHK Apartment for Sale in Hinjawadi Pune
273,1 BHK Apartment for Sale in Camp Pune


These examples reveal an important pattern: most titles without commas end with Pune, but the locality can contain multiple words.

For example:
- 2 BHK Apartment for Sale in Wakad Pune → Wakad
- 3 BHK Apartment for Sale in Baner Pashan Link Road Pune → Baner Pashan Link Road
- 2 BHK Apartment for Sale in Bhumkar Nagar Wakad Pune → potentially Bhumkar Nagar Wakad
- 2 BHK Apartment for Sale in Kohinoor Pune → ambiguous: Kohinoor could be a project name rather than a locality.
So we shouldn't simply take the last word before Pune as the locality.

In [128]:
with pd.option_context("display.max_colwidth", None):
    display(
        real_estate.loc[
            real_estate["Title"].str.contains(",", regex=False),
            ["Title"]
        ].sample(25, random_state=42)
    )

,Title
722,"1 BHK Apartment for Sale in Sadaphuli, Talegaon Dabhade Pune"
386,"4 BHK Penthouse for Sale in Marvel Zephyr, Kharadi Pune"
112,"4 BHK Apartment for Sale in Marvel Ribera, Bund Garden, Sangamvadi Pune"
737,"2 BHK Apartment for Sale in Peninsula Address One, Gahunje Pune"
39,"4 BHK Apartment for Sale in Nyati Elenor, Bibwewadi Pune"
66,"3 BHK Apartment for Sale in Tayal K Town Kiwale, Kiwale, Dehu Road Pune"
309,"2 BHK Apartment for Sale in Venkatesh Serenity, Dhayari Phata Road Pune"
271,"3 BHK Penthouse for Sale in ARV Royale, Hadapsar Pune"
701,"3 BHK Apartment for Sale in Kolte Patil Maestros, Wanowrie Pune"
920,"2 BHK Apartment for Sale in Woodsville Phase I, Moshi Pune"


Now the pattern is clearer. Most comma-containing titles follow this structure:

[Property details] in [Society/Project], [Locality] Pune

In [129]:
real_estate["Location_Text"] = real_estate["Title"].str.extract(
    r"for Sale in\s+(.+)",
    expand=False # To return it as Series instead of DF 
)

In [130]:
with pd.option_context("display.max_colwidth", None):
    display(real_estate[["Title", "Location_Text"]].sample(25, random_state=42))

,Title,Location_Text
166,"3 BHK Apartment for Sale in VTP MONARQUE, Hinjawadi Pune","VTP MONARQUE, Hinjawadi Pune"
433,"3 BHK Apartment for Sale in ARV Regalia, NIBM Annexe Area Pune","ARV Regalia, NIBM Annexe Area Pune"
30,"2 BHK Apartment for Sale in Jasminum, Magarpatta Pune","Jasminum, Magarpatta Pune"
67,"3 BHK Apartment for Sale in GK Sindhu Skies, Pimple Saudagar, Pimpri Chinchwad Pune","GK Sindhu Skies, Pimple Saudagar, Pimpri Chinchwad Pune"
752,"4 BHK Apartment for Sale in Livience Aleenta, Pashan Pune","Livience Aleenta, Pashan Pune"
886,2 BHK Apartment for Sale in Laxmi Nagar Balewadi Pune,Laxmi Nagar Balewadi Pune
365,"3 BHK Penthouse for Sale in Lunkad Orchids, Clover Park, Viman Nagar Pune","Lunkad Orchids, Clover Park, Viman Nagar Pune"
107,"4 BHK Apartment for Sale in ANP Autograph, Punawale, Pimpri Chinchwad Pune","ANP Autograph, Punawale, Pimpri Chinchwad Pune"
317,"3 BHK Apartment for Sale in Rachana Bella Casa, Baner Pune","Rachana Bella Casa, Baner Pune"
295,"2 BHK Apartment for Sale in Eiffel City Phase II, Kharabwadi Pune","Eiffel City Phase II, Kharabwadi Pune"


| Location_Text                                           | Likely Locality                   |
| ------------------------------------------------------- | --------------------------------- |
| VTP MONARQUE, Hinjawadi Pune                            | Hinjawadi                         |
| ARV Regalia, NIBM Annexe Area Pune                      | NIBM Annexe Area                  |
| GK Sindhu Skies, Pimple Saudagar, Pimpri Chinchwad Pune | Pimple Saudagar, Pimpri Chinchwad |
| Erandwane Pune                                          | Erandwane                         |
| Narhe Road Pune                                         | Narhe Road                        |
| Jasminum, Jasminium Society Pune                        | Ambiguous                         |

logic:
- If commas exist, take the text after the first comma.
- If no comma exists, keep the full location text for manual review.
- Remove the trailing Pune from the candidate.

In [131]:
def location_extractor(location_text):
    location_text = location_text.removesuffix(" Pune")

    if "," not in location_text:
        return location_text.strip()
    result = ", ".join(part.strip() for part in location_text.split(",")[1:])

    return result

real_estate.Location_Text.apply(location_extractor).sample(10, random_state=42)

166                            Hinjawadi
433                     NIBM Annexe Area
30                            Magarpatta
67     Pimple Saudagar, Pimpri Chinchwad
752                               Pashan
886                 Laxmi Nagar Balewadi
365             Clover Park, Viman Nagar
107           Punawale, Pimpri Chinchwad
317                                Baner
295                           Kharabwadi
Name: Location_Text, dtype: str

In [132]:
real_estate["Locality"] = real_estate["Location_Text"].apply(location_extractor)

In [133]:
real_estate["Locality"] = (
    real_estate["Locality"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [134]:
real_estate["Locality"].value_counts(dropna= False).head(30)

Locality
Kharadi                              64
Wakad                                36
Baner                                35
Hinjawadi                            34
Undri                                28
Bavdhan                              26
Balewadi                             25
Hadapsar                             25
NIBM Road                            19
Mundhwa                              19
Magarpatta                           16
Tathawade Pimpri Chinchwad           16
Wagholi                              15
Pimple Saudagar, Pimpri Chinchwad    14
Punawale, Pimpri Chinchwad           13
EON Free Zone, Kharadi               13
Keshav Nagar Mundhwa                 12
Mahalunge                            12
Kothrud                              12
Viman Nagar                          11
Amanora Park Town                    10
Pashan                               10
Kondhwa                              10
Kalyani Nagar                         9
Ravet, Pimpri Chinchwad        

In [135]:
candidates = [
    "Tathawade Pimpri Chinchwad",
    "EON Free Zone, Kharadi",
    "Keshav Nagar Mundhwa",
    "Amanora Park Town",
    "Bhugaon, Bavdhan",
]

with pd.option_context("display.max_colwidth", None):
    display(
        real_estate.loc[
            real_estate["Locality"].isin(candidates),
            ["Title", "Location_Text", "Locality"]
        ].sort_values("Locality")
    )

,Title,Location_Text,Locality
372,"4 BHK Apartment for Sale in Amanora Neo Towers, Amanora Park Town Pune","Amanora Neo Towers, Amanora Park Town Pune",Amanora Park Town
404,2 BHK Apartment for Sale in Amanora Park Town Pune,Amanora Park Town Pune,Amanora Park Town
49,"4 BHK Apartment for Sale in Amanora Gateway Towers 2, Amanora Park Town Pune","Amanora Gateway Towers 2, Amanora Park Town Pune",Amanora Park Town
421,"2 BHK Apartment for Sale in Amanora Sterling Towers R4, Amanora Park Town Pune","Amanora Sterling Towers R4, Amanora Park Town Pune",Amanora Park Town
64,"3 BHK Apartment for Sale in Amanora Elevate Towers, Amanora Park Town Pune","Amanora Elevate Towers, Amanora Park Town Pune",Amanora Park Town
820,"3 BHK Apartment for Sale in Amanora Elevate Towers, Amanora Park Town Pune","Amanora Elevate Towers, Amanora Park Town Pune",Amanora Park Town
806,"2 BHK Apartment for Sale in Amanora Neo Towers, Amanora Park Town Pune","Amanora Neo Towers, Amanora Park Town Pune",Amanora Park Town
468,"3 BHK Apartment for Sale in Amanora Gateway Towers, Amanora Park Town Pune","Amanora Gateway Towers, Amanora Park Town Pune",Amanora Park Town
778,"4 BHK Apartment for Sale in Amanora Elevate Towers, Amanora Park Town Pune","Amanora Elevate Towers, Amanora Park Town Pune",Amanora Park Town
435,"3 BHK Apartment for Sale in Amanora Future Towers, Amanora Park Town Pune","Amanora Future Towers, Amanora Park Town Pune",Amanora Park Town


In [136]:
real_estate.loc[
    real_estate.Locality.str.contains('Pimpri Chinchwad'),
    'Locality'
].value_counts()

Locality
Tathawade Pimpri Chinchwad                16
Pimple Saudagar, Pimpri Chinchwad         14
Punawale, Pimpri Chinchwad                13
Ravet, Pimpri Chinchwad                    9
Pimpri Chinchwad                           9
Thergaon, Pimpri Chinchwad                 3
Pimple Nilakh, Pimpri Chinchwad            3
Pimple Gurav, Pimpri Chinchwad             2
Siddhivinayak Ginger, Pimpri Chinchwad     1
Chinchwad, Pimpri Chinchwad                1
Kalewadi Pimpri Chinchwad                  1
Name: count, dtype: int64

TO:DO

### Area_Measurement_Type

In [137]:
real_estate['Area_Measurement_Type'] = np.where(
    real_estate.Carpet_Area.notna(),
    'Carpet Area',
    'Super Area'
)

### Price_Per_Cr

In [138]:
real_estate["Price_Per_Cr"] = real_estate.Price / 10_000_000

In [139]:
real_estate['Locality'] = real_estate.Locality.str.strip().str.strip(',')

In [140]:
real_estate.head()

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Transaction,Furnishing,facing,Society,Ownership,...,Floor_Label,Overlooks_Garden_Park,Overlooks_Pool,Overlooks_Main_Road,BHK,Property_Type,Location_Text,Locality,Area_Measurement_Type,Price_Per_Cr
0,3 BHK Flat for Sale in Saarrthi Shimmer n Shi...,17500000.0,11305.0,1165.0,Ready to Move,Resale,Furnished,East,Saarrthi Shimmer n Shine,Co-operative Society,...,NaN,False,False,True,3,Flat,"Saarrthi Shimmer n Shine, Sinhgad Road, Pune",Sinhgad Road,Carpet Area,1.750
1,"3 BHK Flat for Sale in Acolade, Kharadi, Pune",15200000.0,8444.0,1400.0,Ready to Move,Resale,Semi-Furnished,East,Acolade,Freehold,...,NaN,True,False,False,3,Flat,"Acolade, Kharadi, Pune",Kharadi,Carpet Area,1.520
2,"4 BHK Flat for Sale in Windsor Avenue, Wanowr...",26000000.0,11064.0,NaN,Ready to Move,Resale,Semi-Furnished,East,Windsor Avenue,Co-operative Society,...,NaN,False,False,True,4,Flat,"Windsor Avenue, Wanowrie, Pune",Wanowrie,Super Area,2.600
3,"1 BHK Flat for Sale in Raheja Sterling, NIBM ...",6160000.0,8107.0,560.0,Ready to Move,Resale,Unfurnished,East,Raheja Sterling,Co-operative Society,...,NaN,True,False,True,1,Flat,"Raheja Sterling, NIBM Road, Pune",NIBM Road,Carpet Area,0.616
4,"3 BHK House for Sale in Kalpana Empress, Wano...",27500000.0,13750.0,NaN,Ready to Move,Resale,Furnished,East,Kalpana Empress,Co-operative Society,...,NaN,False,False,True,3,House,"Kalpana Empress, Wanowrie Pune",Wanowrie,Super Area,2.750


### Price_Tier

| Tier      | Price Range                |
| --------- | -------------------------- |
| Budget    | ₹0 to below ₹50 lakh       |
| Mid-Range | ₹50 lakh to below ₹1 crore |
| Premium   | ₹1 crore and above         |

In [141]:
real_estate["Price_Tier"] = pd.cut(
    real_estate["Price"],
    bins=[0, 5_000_000, 10_000_000, float("inf")],
    labels=["Budget", "Mid-Range", "Premium"],
    right=False, #include left, exclude right
    include_lowest=True
)

In [142]:
real_estate["Price_Tier"].value_counts()

Price_Tier
Premium      624
Mid-Range    240
Budget        63
Name: count, dtype: int64

In [143]:
real_estate["Price_Tier"].isna().sum()

np.int64(0)

### Price_Outlier_Flag

In [144]:
real_estate["Price_Outlier_Flag"] = (
    (real_estate["Price"] >= 100_000_000) |
    (real_estate["Rate_per_SqFt"] >= 50_000)
)

In [145]:
real_estate["Price_Outlier_Flag"].value_counts()

Price_Outlier_Flag
False    924
True       3
Name: count, dtype: int64

In [146]:
real_estate.loc[
    real_estate["Price_Outlier_Flag"],
    ["Title", "Price", "Rate_per_SqFt", "BHK", "Locality"]
]

,Title,Price,Rate_per_SqFt,BHK,Locality
89,3 BHK Apartment for Sale in Majestique Evolvus...,163800000.0,150000.0,3,Kharadi
469,"4 BHK Apartment for Sale in Rohan 1 Modibaug, ...",149900000.0,24374.0,4,Shivajinagar
876,"4 BHK Apartment for Sale in Yoo Pune, Magarpat...",105500000.0,20095.0,4,Magarpatta


In [147]:
real_estate[
    ["BHK", "Property_Type", "Locality",
     "Area_Measurement_Type", "Price_Per_Cr", "Price_Tier"]
].isna().sum()

BHK                      0
Property_Type            0
Locality                 0
Area_Measurement_Type    0
Price_Per_Cr             0
Price_Tier               0
dtype: int64

### Clean up

In [148]:
real_estate.drop(columns=["Location_Text"], inplace=True)

In [149]:
real_estate.info()

<class 'pandas.DataFrame'>
Index: 927 entries, 0 to 929
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   Title                  927 non-null    str     
 1   Price                  927 non-null    float64 
 2   Rate_per_SqFt          820 non-null    float64 
 3   Carpet_Area            843 non-null    float64 
 4   Status                 735 non-null    str     
 5   Transaction            927 non-null    str     
 6   Furnishing             926 non-null    str     
 7   facing                 776 non-null    str     
 8   Society                808 non-null    str     
 9   Ownership              776 non-null    str     
 10  Car_Parking            554 non-null    str     
 11  Bathroom               927 non-null    int64   
 12  Balcony                864 non-null    float64 
 13  Super_Area             84 non-null     float64 
 14  Under_Construction     178 non-null    str     
 15  Floor

---

## Step 5 — Export the cleaned dataset

In [150]:
real_estate.to_csv("../data/cleaned/clean_rent_data.csv", index=False)

### Verification

In [151]:
cleaned_check = pd.read_csv("../data/cleaned/clean_rent_data.csv")

print("Exported shape:", cleaned_check.shape)
print("Columns match:", list(cleaned_check.columns) == list(real_estate.columns))

Exported shape: (927, 28)
Columns match: True


In [152]:
del cleaned_check